# Tutorial on time of arrival estimation under jitter noise

## Introduction

Time of Arrival (TOA) estimation is a fundamental problem in localization. It involves determining the precise moment a signal, or a characteristic event within a signal, arrives at a sensor or detector.

However, signals are often corrupted by various forms of uncertainty, collectively known as **jitter**. Jitter can be random or deterministic deviations in the timing of events from their ideal or expected instances. These timing variations can significantly degrade the accuracy of TOA estimates and, consequently, the performance of systems that rely on them.

This tutorial explores the concepts of correlation, its application in TOA estimation, and how **random** jitter influences this process. We will delve into the mathematical foundations and analyze the effects of applying cross-correlation for TOA estimation under random jitter.

## Introduction to timing Jitter

### Types of Jitter: Random, Deterministic, and Periodic

Timing jitter for signal processing can be classified into 3 main categories based on its statistical properties and underlying causes. This document primarily focuses on random jitter.

---

#### 1. Random Jitter (RJ)

**Characteristics:**
Random jitter is unbounded and typically modeled by a gaussian distribution, as its primary cause is due to thermal noise.

**Mathematical Model:**

For a signal with TOA of $t_0$, the actual TOA with random jitter is:

$$
T_{\text{actual}} = T + \Delta t_{RJ}
$$

where $\Delta t_{RJ} \sim \mathcal{N}(0, \sigma_{RJ}^2)$

---

#### 2. Deterministic Jitter (DJ)

Can be bounded or unbounded, predictable, periodic or not periodic, and may be reduced through equalization or calibration. Primary causes include crosstalk interference, 50/60 Hz power supply noise, and multipath.
Periodic jitter especially is easily visible in frequency domain.



### Effects of Jitter on Timing Problems

#### Impact of Thermal Noise

Thermal noise causes the correlation peak to fluctuate around the true signal arrival time. The key effects are:

**1. Peak Location Variance:** ## NEEDS RESOURCE
- The detected peak time $\hat{t}_0$ is a random variable distributed around the true arrival time $t_0$
- For high SNR and Gaussian noise: $\hat{t}_0 \sim \mathcal{N}(t_0, \sigma_t^2)$
- The standard deviation $\sigma_t$ represents the timing jitter

**2. SNR-Dependent Jitter:**
- Lower SNR → larger timing uncertainty
- The Cramér-Rao lower bound shows that timing variance is inversely proportional to SNR ## TODO NEEDS RESOURCE

**3. Bandwidth-Dependent Resolution:**
- Wider bandwidth signals → sharper correlation peaks → better timing resolution
- Narrowband signals have broader correlation peaks, making peak localization less precise
- Timing variance is inversely proportional to the square of the signal bandwidth

---

#### Combined Effects on Timing Accuracy

The observed timing jitter results from multiple sources:

1. **Thermal noise** perturbing the correlation peak location
2. **Other sources** (discussed in next section)

The relative importance of each source depends on the specific system parameters and operating environment.


### Other Causes of Jitter
## TODO NEEDS REVIEW

#### 1. Multipath Propagation

Multipath refers to the phenomenon where the transmitted signal reaches the receiver via multiple paths due to reflections, creating multiple delayed and attenuated copies of the signal.

**Mathematical Model:**

The received signal under multipath conditions:

$$
r(t) = \sum_{k=0}^{K-1} \alpha_k s(t - t_0 - \tau_k) + n(t)
$$

where:
- $\alpha_k$ is the attenuation coefficient of path $k$
- $\tau_k$ is the relative delay of path $k$
- $K$ is the number of propagation paths

**Effect on Correlation:**

The correlation function becomes:

$$
R_{rs}(\tau) = \sum_{k=0}^{K-1} \alpha_k R_{ss}(\tau - t_0 - \tau_k) + N(\tau)
$$

This creates multiple peaks or a distorted peak shape, degrading timing estimation accuracy.

---

#### 2. Interference - Non-random, non-SOI Waveforms

**Description:**
- Different propagating signals (other brain activity, artifacts) can affect array electrodes differently
- Unlike thermal noise, interference is structured and may have spatial coherence

**Potential Mitigation Techniques:**

**MVDR (Minimum Variance Distortionless Response):**
- Adaptive beamforming technique that attempts to minimize output variance while maintaining unity gain in the direction of interest
- Requires knowledge or estimation of the signal direction and interference/noise covariance

**MUSIC (Multiple Signal Classification):**
- Uses eigendecomposition to separate signal and noise subspaces
- Can estimate direction of arrival for multiple signals
- Requires multiple snapshots and assumptions about the number of signal sources

**Note:** These techniques require careful consideration of the specific signal model and may not be directly applicable without modification.

#### 3. Synchronization Errors

**Description:**
- In communication systems, timing recovery algorithms attempt to align receiver timing with transmitter
- Imperfect estimation leads to residual timing errors

#### 4. Oscillator Instability

**Description:**
- Clocks in signal processing systems can drift over time
- Different oscillators can drift relative to each other (clock skew)
- Frequency instability creates time-varying sampling rates

**Mitigation:**
- Using a common clock for all channels in a multi-channel system eliminates relative drift between channels
- Only absolute drift remains, which may be less critical for some applications

#### 5. ADC Aperture Jitter

**Description:**
- Analog-to-Digital Converters can sample at slightly incorrect times due to aperture jitter
- The sampling clock itself has phase noise
- Creates uncertainty in the actual sampling instant

**Quantifying ADC Jitter:**

The timing uncertainty due to ADC jitter adds variance:

$$
\sigma_{\text{total}}^2 = \sigma_t^2 + \sigma_{\text{ADC}}^2
$$

**Relevance:**
- ADC jitter is primarily significant for high-frequency signals or high sampling rate systems
- For low-bandwidth signals (< 10 kHz), typical ADC aperture jitter (1-100 ps) is generally negligible
- Impact increases linearly with signal frequency

---

#### Side Notes

**Question for Future Exploration:**  
How does electrode degradation affect signal collection - could it be frequency selective?

- Electrode impedance may increase with degradation
- Could act as a frequency-dependent filter
- May affect high-frequency components more than low-frequency components
- Worth investigating if electrode aging impacts detection performance

## What Is Correlation?

### Fundamental Concept

Correlation measures the similarity between two signals as a function of time lag. It quantifies how much one signal resembles another (or itself) when shifted by a delay $$\tau$$.

**Mathematical Definition:**

For continuous-time signals $$r(t)$$ and $$s(t)$$, the cross-correlation function is:

$$
R_{rs}(\tau) = \int_{-\infty}^{\infty} r(t) s^*(t - \tau) \, dt
$$

For discrete-time signals (sampled data):

$$
R_{rs}[m] = \sum_{n=-\infty}^{\infty} r[n] s^*[n - m]
$$

where $$s^*$$ denotes the complex conjugate (for real signals, $$s^* = s$$).

**Why Correlation for Time-of-Arrival Problems:**

The chain of reasoning is:
1. **Correlation** provides a similarity measure that peaks when signals are best aligned
2. **Peak detection** identifies the time lag where this maximum similarity occurs
3. **Timing offset estimation** uses the peak location to determine the relative delay between signals
4. **Time-of-arrival (ToA)** problems require knowing when a signal arrived, which is directly given by this timing offset

---

### Auto-Correlation: The Ground Truth Reference

**Definition:**

Auto-correlation correlates a signal with a delayed version of itself:

$$
R_{ss}(\tau) = \int_{-\infty}^{\infty} s(t) s(t - \tau) \, dt
$$

**Why Auto-Correlation Matters:**

In the **absence of noise**, auto-correlation represents the ideal correlation function:
- It shows the inherent timing resolution of the signal
- The peak width reveals the signal's bandwidth characteristics
- It serves as a **ground truth** for what perfect correlation looks like

**Key Properties:**

1. **Symmetry:** $$R_{ss}(\tau) = R_{ss}(-\tau)$$ for real signals
2. **Maximum at zero lag:** $$R_{ss}(0) \geq |R_{ss}(\tau)|$$ for all $$\tau$$
   - $$R_{ss}(0)$$ equals the signal energy
3. **Related to power spectral density:** Via the Wiener-Khinchin theorem:
   $$S_{ss}(f) = \mathcal{F}\{R_{ss}(\tau)\}$$

**Applications:**

- **Periodicity detection:** Periodic signals show peaks at multiples of the period
- **Bandwidth estimation:** The width of the main lobe indicates signal bandwidth
- **Baseline for detection performance:** Comparison standard for noisy measurements

---

### Cross-Correlation: Scoring Against the Ground Truth

**Definition:**

Cross-correlation measures similarity between two different signals:

$$
R_{rs}(\tau) = \int_{-\infty}^{\infty} r(t) s(t - \tau) \, dt
$$

**Interpretation as a Score:**

When $$r(t) = s(t - t_0) + n(t)$$ (signal plus noise):

$$
R_{rs}(\tau) = R_{ss}(\tau - t_0) + R_{ns}(\tau)
$$

- The cross-correlation attempts to match the **ground truth** auto-correlation $$R_{ss}(\tau)$$
- The peak location estimates the true delay $$t_0$$
- The peak magnitude compared to $$R_{ss}(0)$$ indicates detection quality
- Noise $$R_{ns}(\tau)$$ degrades this match

**Normalized Cross-Correlation:**

To remove amplitude dependence and create a proper score:

$$
\rho_{rs}(\tau) = \frac{R_{rs}(\tau)}{\sqrt{R_{rr}(0) \cdot R_{ss}(0)}}
$$

This ensures $$-1 \leq \rho_{rs}(\tau) \leq 1$$, where:
- $$\rho = 1$$ indicates perfect match
- $$\rho = 0$$ indicates no correlation
- Values close to auto-correlation peak indicate good signal detection

---

### Peak Detection and Time Offset Estimation

#### Basic Peak Detection

The time lag that maximizes the cross-correlation function provides the time-of-arrival estimate:

$$
\hat{\tau} = \arg\max_{\tau} R_{rs}(\tau)
$$

For sampled signals, compute correlation at discrete lags and find the maximum:

$$
\hat{m} = \arg\max_{m} R_{rs}[m]
$$

Then convert to time: $$\hat{\tau} = \hat{m} \cdot T_s$$ where $$T_s$$ is the sampling period.

---

#### Correlation Peak Width and Bandwidth

The width of the correlation peak determines timing resolution:

**Relationship to Bandwidth:**

The main lobe width of $$R_{ss}(\tau)$$ is inversely proportional to signal bandwidth $$B$$:

$$
\Delta \tau_{\text{width}} \approx \frac{1}{B}
$$

**Implications:**

- **Wideband signals** (large $$B$$) → narrow correlation peak → precise timing
- **Narrowband signals** (small $$B$$) → broad correlation peak → poor timing resolution
- This is why GPS and radar use wideband or spread-spectrum signals

**Example:**
- 1 kHz bandwidth signal: $$\Delta \tau \approx 1$$ ms resolution
- 10 MHz bandwidth signal: $$\Delta \tau \approx 0.1$$ μs resolution

---

#### Noise Effects on Peak Detection

**In Presence of Noise:**

If $$r(t) = s(t - t_0) + n(t)$$ where $$n(t)$$ is noise:

$$
R_{rs}(\tau) = R_{ss}(\tau - t_0) + R_{ns}(\tau)
$$

The noise term $$R_{ns}(\tau)$$ causes the peak location to fluctuate, introducing timing jitter.

**Timing Variance:**

The Cramér-Rao lower bound for timing estimation variance is:

$$
\sigma_{\tau}^2 \geq \frac{1}{8\pi^2 \beta^2 \cdot \text{SNR}}
$$

where $$\beta^2$$ is the mean-square bandwidth of the signal:

$$
\beta^2 = \frac{\int_{-\infty}^{\infty} f^2 |S(f)|^2 \, df}{\int_{-\infty}^{\infty} |S(f)|^2 \, df}
$$

**Key Insights:**
- Timing accuracy improves with higher SNR
- Timing accuracy improves with larger bandwidth (higher $$\beta$$)
- There is a fundamental limit imposed by noise

---

#### Threshold Detection

To distinguish true peaks from noise fluctuations:

**Setting Thresholds:**
- Set threshold based on expected SNR and noise statistics
- Use statistical detection theory (e.g., Neyman-Pearson criterion)
- False alarm rate depends on threshold choice

**Trade-offs:**
- **High threshold:** Fewer false alarms, but may miss weak signals (missed detections)
- **Low threshold:** Better detection of weak signals, but more false alarms
- Optimal threshold balances these competing concerns based on application requirements

---

#### Practical Considerations for Noise-Induced Jitter

**1. Averaging Multiple Measurements:**

When multiple independent observations are available:

$$
\sigma_{\tau,\text{avg}}^2 = \frac{\sigma_{\tau}^2}{N}
$$

Averaging $$N$$ measurements reduces timing jitter by factor $$\sqrt{N}$$.

**2. SNR Estimation:**

Knowing the noise level helps set appropriate detection thresholds:
- Estimate noise from signal-free regions
- Use known signal characteristics to predict expected peak height
- Adaptive thresholds adjust based on local noise conditions

**3. Peak Quality Metrics:**

Beyond simple peak detection, assess peak quality:
- **Peak-to-sidelobe ratio:** Higher ratios indicate cleaner detection
- **Peak sharpness:** Narrower peaks suggest less noise corruption
- **Confidence intervals:** Quantify uncertainty in timing estimate based on peak shape

### Cross-Correlation vs. Auto-Correlation

### Peak Detection and Time Offset Estimation

## How Jitter Affects Correlation

### Jitter Effects on Correlation Peaks

### Observing Correlation Degradation

### Estimating Jitter with Correlation Techniques

# Case Study: Correlation between channels to detect neurological signals and signatures that exist between channels